##Task 1-Tensors

In [1]:
import numpy as np
import torch

# ---- 1. Creating tensors ----
t_list   = torch.tensor([[1, 2, 3], [4, 5, 6]])       # from a Python list
t_zeros  = torch.zeros(2, 3)
t_ones   = torch.ones(2, 3)
t_arange = torch.arange(0, 12)                        # 0, 1, ..., 11
t_randn  = torch.randn(3, 4)                          # random, mean 0, std 1

for name, t in [("tensor", t_list), ("zeros", t_zeros), ("ones", t_ones),
                ("arange", t_arange), ("randn", t_randn)]:
    print(f"{name:<7} shape {str(tuple(t.shape)):<8} dtype {str(t.dtype):<14} device {t.device}")

# ---- 2. Slicing, reshaping, transposing ----
a = torch.arange(12).reshape(3, 4)
print("\na =\n", a)
print("Row 1:          ", a[1])
print("Column 2:       ", a[:, 2])
print("Rows 0-1, cols 1-2:\n", a[0:2, 1:3])
print("view(4, 3):\n", a.view(4, 3))
print("reshape(2, 6):\n", a.reshape(2, 6))
print("Transpose a.T shape:", a.T.shape)

# ---- 3. Matrix multiplication (3x4) @ (4x2) ----
A = torch.randn(3, 4)
B = torch.randn(4, 2)
print(f"\n(3x4) @ (4x2) -> {(A @ B).shape}")
print(f"torch.matmul    -> {torch.matmul(A, B).shape}")
print("Same result:", torch.allclose(A @ B, torch.matmul(A, B)))

# ---- 4. NumPy <-> tensor ----
np_arr = np.array([1.0, 2.0, 3.0])
t_from_np = torch.from_numpy(np_arr)
back_to_np = t_from_np.numpy()
print("\nNumPy -> tensor:", t_from_np, "| dtype", t_from_np.dtype)
print("tensor -> NumPy:", back_to_np, "| type", type(back_to_np).__name__)
np_arr[0] = 99                                     # they share memory!
print("After changing the NumPy array, the tensor is:", t_from_np)

# ---- 5. Device: GPU if available, else CPU ----
device = "cuda" if torch.cuda.is_available() else "cpu"
x = torch.randn(3, 3).to(device)
print(f"\nGPU available: {torch.cuda.is_available()} | Device: {x.device}")

tensor  shape (2, 3)   dtype torch.int64    device cpu
zeros   shape (2, 3)   dtype torch.float32  device cpu
ones    shape (2, 3)   dtype torch.float32  device cpu
arange  shape (12,)    dtype torch.int64    device cpu
randn   shape (3, 4)   dtype torch.float32  device cpu

a =
 tensor([[ 0,  1,  2,  3],
        [ 4,  5,  6,  7],
        [ 8,  9, 10, 11]])
Row 1:           tensor([4, 5, 6, 7])
Column 2:        tensor([ 2,  6, 10])
Rows 0-1, cols 1-2:
 tensor([[1, 2],
        [5, 6]])
view(4, 3):
 tensor([[ 0,  1,  2],
        [ 3,  4,  5],
        [ 6,  7,  8],
        [ 9, 10, 11]])
reshape(2, 6):
 tensor([[ 0,  1,  2,  3,  4,  5],
        [ 6,  7,  8,  9, 10, 11]])
Transpose a.T shape: torch.Size([4, 3])

(3x4) @ (4x2) -> torch.Size([3, 2])
torch.matmul    -> torch.Size([3, 2])
Same result: True

NumPy -> tensor: tensor([1., 2., 3.], dtype=torch.float64) | dtype torch.float64
tensor -> NumPy: [1. 2. 3.] | type ndarray
After changing the NumPy array, the tensor is: tensor([99.,  2., 

## Task 2 - Autograd

In [2]:
# ---- 1. Autograd reproduces the hand calculation ----
w = torch.tensor(1.0, requires_grad=True)
b = torch.tensor(0.0, requires_grad=True)
loss = (w * 2 + b - 10) ** 2
loss.backward()                                    # chain rule, done automatically
print(f"loss = {loss.item()} | w.grad = {w.grad.item()}, b.grad = {b.grad.item()}")
print("Hand calculation:      w.grad = -32, b.grad = -16")

# ---- 2. Gradient descent with autograd (30 steps) ----
def run_gd(zero_grads=True, steps=30, lr=0.05):
    w = torch.tensor(1.0, requires_grad=True)
    b = torch.tensor(0.0, requires_grad=True)
    history = []
    for step in range(steps):
        loss = (w * 2 + b - 10) ** 2
        loss.backward()                            # adds new gradients into w.grad / b.grad
        with torch.no_grad():                      # update without tracking it as part of the graph
            w -= lr * w.grad
            b -= lr * b.grad
        if zero_grads:
            w.grad.zero_()                         # clear for the next step
            b.grad.zero_()
        history.append(loss.item())
    return w.item(), b.item(), history

w_ok, b_ok, hist_ok = run_gd(zero_grads=True)
print("\nWith zeroing (correct):")
for s in [0, 5, 10, 20, 29]:
    print(f"  step {s:>2}: loss {hist_ok[s]:.6f}")
print(f"  final w = {w_ok:.3f}, b = {b_ok:.3f}  ->  w*2 + b = {2*w_ok + b_ok:.3f}")

# ---- 3. What happens if you forget to zero the gradients ----
w_bad, b_bad, hist_bad = run_gd(zero_grads=False)
print("\nWithout zeroing (bug):")
for s in [0, 5, 10, 20, 29]:
    print(f"  step {s:>2}: loss {hist_bad[s]:.6f}")
print(f"  final w = {w_bad:.3f}, b = {b_bad:.3f}  ->  w*2 + b = {2*w_bad + b_bad:.3f}")

# Simple illustration: two backward() calls without zeroing ADD UP
w = torch.tensor(1.0, requires_grad=True)
b = torch.tensor(0.0, requires_grad=True)
for i in range(2):
    ((w * 2 + b - 10) ** 2).backward()
    print(f"After backward() call {i+1}: w.grad = {w.grad.item()}")   # -32, then -64

# ---- 4. detach() ----
y = w * 3
print("\ny.requires_grad:", y.requires_grad, "| y.detach().requires_grad:", y.detach().requires_grad)

loss = 64.0 | w.grad = -32.0, b.grad = -16.0
Hand calculation:      w.grad = -32, b.grad = -16

With zeroing (correct):
  step  0: loss 64.000000
  step  5: loss 0.062500
  step 10: loss 0.000061
  step 20: loss 0.000000
  step 29: loss 0.000000
  final w = 4.200, b = 1.600  ->  w*2 + b = 10.000

Without zeroing (bug):
  step  0: loss 64.000000
  step  5: loss 33.062500
  step 10: loss 5.027405
  step 20: loss 28.839552
  step 29: loss 44.902458
  final w = 7.616, b = 3.308  ->  w*2 + b = 18.541
After backward() call 1: w.grad = -32.0
After backward() call 2: w.grad = -64.0

y.requires_grad: True | y.detach().requires_grad: False


In [3]:
import time

size = 2000
A = torch.randn(size, size)
B = torch.randn(size, size)

A @ B                                   # warm-up run (first call is slower)
start = time.time()
for _ in range(5):
    C = A @ B
cpu_time = (time.time() - start) / 5
print(f"CPU: {size}x{size} @ {size}x{size} takes {cpu_time*1000:.1f} ms")

if torch.cuda.is_available():
    Ag, Bg = A.to("cuda"), B.to("cuda")
    Ag @ Bg
    torch.cuda.synchronize()            # GPU runs asynchronously; wait for it
    start = time.time()
    for _ in range(5):
        Cg = Ag @ Bg
    torch.cuda.synchronize()
    gpu_time = (time.time() - start) / 5
    print(f"GPU: {gpu_time*1000:.1f} ms  ->  {cpu_time/gpu_time:.0f}x faster")
else:
    print("No GPU on this laptop. On Google Colab (Runtime -> Change runtime type -> T4 GPU), "
          "the same multiplication typically runs 10-50x faster.")

CPU: 2000x2000 @ 2000x2000 takes 98.2 ms
No GPU on this laptop. On Google Colab (Runtime -> Change runtime type -> T4 GPU), the same multiplication typically runs 10-50x faster.


### Results
- Tensors: (3x4) @ (4x2) -> torch.Size([3, 2]); Device: cpu (no NVIDIA GPU on this laptop).
- Autograd: w.grad = -32.0, b.grad = -16.0 - matches the hand calculation (2*(2-10)*2 = -32, 2*(2-10) = -16).
- Gradient descent with autograd (30 steps, lr 0.05): loss 64 -> 0, w = 4.2, b = 1.6, w*2 + b = 10.
- Forgetting grad.zero_(): gradients add up (-32, then -64 after two backward() calls), so updates overshoot; loss 64 -> 5 -> 44.9 and w*2 + b ends at 18.5 instead of 10.

### requires_grad and detach()
1. `requires_grad=True` tells PyTorch to record every operation on that tensor so `.backward()` can compute its gradient (used for weights and biases).
2. `detach()` returns a copy of the tensor that is cut off from this record, so no gradient flows back through it (useful for logging values, plotting, or converting to NumPy).
3. `with torch.no_grad():` switches recording off for a whole block - used for weight updates and evaluation, where gradients are not needed.